In [1]:
from flappy_agent import *
import numpy as np

pygame 2.6.1 (SDL 2.28.4, Python 3.13.0)
Hello from the pygame community. https://www.pygame.org/contribute.html
couldn't import doomish


/Users/arnarfreyrerlingsson/Documents/PPO Games/.venv/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
# Fresh agent
agent = FlappyAgent([256,256,256])
# Or load existing weights to continue training:
#w_path='flappy weights/flappy_ppo_64x64x64.pt'
#agent = FlappyAgent(hidden_layers=[1024, 128, 16,256,256],weights_path=w_path)

In [3]:
training_params = {
    'gamma': 0.99,
    'lam': 0.95,
    'clip_eps': 0.2,
    'clip_coef': 1.0,
    'value_coef': 0.5,
    'entropy_coef': 0.01,
    'max_grad_norm': 5.0,
    'learning_rate': 0.0003,
    'ppo_epochs': 5,
    'num_epochs': 5000,
    'target_steps': 8192,
    'minibatch_size': 512,
    'print_freq': 10,
    'ema_alpha': 0.9,
    'value_loss': 'mse',
    'normalize_advantage': True,
    'reward_values': {'positive':1.0, 'tick':0.0, 'loss':-5.0},
}

In [ ]:
print('Using parameters:')
for key, value in training_params.items():
    print(f'{key}: {value}')
print()
agent.run_training(**training_params, test_exploit=True, result_path='Results/NN256x3.csv')

Using parameters:
gamma: 0.99
lam: 0.95
clip_eps: 0.2
clip_coef: 1.0
value_coef: 0.5
entropy_coef: 0.01
max_grad_norm: 5.0
learning_rate: 0.0003
ppo_epochs: 5
num_epochs: 5000
target_steps: 8192
minibatch_size: 512
print_freq: 10
ema_alpha: 0.9
value_loss: mse
normalize_advantage: True
reward_values: {'positive': 1.0, 'tick': 0.0, 'loss': -5.0}


10000


In [ ]:
#torch.save(agent.network.state_dict(), 'flappy weights/flappy_ppo_64x64x64.pt')
print("Weights saved to 'flappy weights/flappy_ppo_trained2.pt'")

In [ ]:
from ple.games.flappybird import FlappyBird
from ple import PLE
game = FlappyBird()
episode = PLE(game, fps=30, display_screen=False, force_fps=True)
episode.init()

agent.prepare_greedy()
agent.play_greedy(episode, max_pipes=None, print_freq=10000)

In [ ]:
#print(states)
for col_name, col_data in states.items():
    mean_val = np.mean(np.array(col_data))
    std_val = np.std(np.array(col_data))
    print(f'{col_name}: {mean_val:.3f} +/- {std_val:.3f}')

In [ ]:
import numpy as np

num_test_episodes = 10
scores = []

from ple.games.flappybird import FlappyBird
from ple import PLE
game = FlappyBird()
episode = PLE(game, fps=30, display_screen=False, force_fps=True)
episode.init()

agent.prepare_greedy()
for ep in range(num_test_episodes):
    pipes = agent.play_greedy(episode, print_freq=1000)
    scores.append(pipes)
    print(f'\nEpisode {ep+1}, pipes {pipes}')

scores = np.array(scores)
print(f"\nAverage pipes: {scores.mean():.1f} +/- {scores.std():.1f}")
print(f"Max pipes: {scores.max()}")
print(f"Min pipes: {scores.min()}")

In [ ]:
import pygame
import os

os.environ.pop('SDL_VIDEODRIVER', None)
pygame.quit()
pygame.init()

num_runs = 10
game_vis = FlappyBird()
p_vis = PLE(game_vis, fps=30, display_screen=True, force_fps=False)
p_vis.init()

for run in range(num_runs):
    p_vis.reset_game()
    total_reward = 0
    while not p_vis.game_over():
        state = state_to_tensor(p_vis.getGameState())
        action, _, _ = agent.get_action(state, mode='Exploit')
        reward = p_vis.act(p_vis.getActionSet()[action])
        total_reward += reward
    print(f"Run {run+1}/{num_runs} - Total reward: {total_reward:.1f}")

pygame.quit()
print("Done.")